# 02b · App RAG empresarial con NVIDIA NIM

Workshop RAG · SI3003 · **Daniel Muñeton**

Esta es mi copia del notebook `02b`. Construye una app RAG autosuficiente:
configura NVIDIA NIM, carga documentos, crea un indice Qdrant temporal y publica
una interfaz Gradio.

Tareas implementadas aqui:
- **A1** · Ejecutar la app de punta a punta (pregunta fuera de corpus incluida).
- **A2** · Mi propio corpus (`data/company_documents.json`) y ruta corregida.
- **A3** · Ingenieria del prompt (version *antes* vs *despues*, documentada).
- **A4** · `build_pipeline` devuelve el vectorstore y `chat()` muestra el score
  de similitud de cada fuente.

> Ejecuta las celdas en orden. Necesitas el secreto `NVIDIA_API_KEY` activo.

## Seccion 1 · Instalar dependencias

In [ ]:
%pip install -qU gradio langchain langchain-core langchain-text-splitters \
    langchain-nvidia-ai-endpoints langchain-qdrant qdrant-client python-dotenv

## Seccion 2 · Configurar la API key

Lee `NVIDIA_API_KEY` desde los Secrets de Colab (o la pide por `getpass`) y
define modelos y nombre de coleccion.

In [ ]:
import os
from getpass import getpass

# En Colab: usa el panel Secrets con el nombre exacto NVIDIA_API_KEY.
try:
    from google.colab import userdata  # type: ignore
    key = userdata.get("NVIDIA_API_KEY")
    if key:
        os.environ["NVIDIA_API_KEY"] = key
except Exception:
    pass

if not os.environ.get("NVIDIA_API_KEY"):
    os.environ["NVIDIA_API_KEY"] = getpass("NVIDIA_API_KEY: ")

from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings

GENERATION_MODEL = "nvidia/nemotron-3-super-120b-a12b"
EMBEDDING_MODEL  = "nvidia/nemotron-3-embed-1b"
COLLECTION       = "company_docs_gradio"

print("\u2705 Configuracion NVIDIA NIM lista")

## Seccion 3 · Cargar los documentos  _(Tarea A2)_

`load_documents` valida que cada documento tenga `id`, `title` y `content`, y
agrega `metadata` vacia si falta.

**Cambio A2:** uso mi propio corpus en `data/company_documents.json` (8
documentos de la empresa ficticia *NebulaCRM*) y corrijo la ruta tal como pide
la guia:

```diff
- company_documents = load_documents(Path("company_documents.json"))
+ company_documents = load_documents(Path("data/company_documents.json"))
```

En Colab, sube la carpeta `data/` o el archivo a la sesion, o clona tu repo:
`!git clone https://github.com/Danimuneton/si3003-workshop-rag-Danimuneton.git`

In [ ]:
import json
from pathlib import Path

DEMO_DOCUMENTS = [
    {"id": "crm_precios", "title": "CloudSync Pro - Precios",
     "content": "El plan Enterprise de CloudSync Pro cuesta $49/mes por usuario con compromiso anual."},
    {"id": "remoto", "title": "Politica de trabajo remoto",
     "content": "Los empleados pueden trabajar remotamente hasta 3 dias por semana, con aprobacion del jefe directo."},
    {"id": "reembolsos", "title": "Politica de reembolsos",
     "content": "Los reembolsos mayores a $100 requieren aprobacion del supervisor y se procesan en 3 a 5 dias habiles."},
    {"id": "onboarding", "title": "Onboarding",
     "content": "Durante la primera semana de onboarding hay modulos obligatorios de seguridad, cumplimiento y cultura."},
]

def load_documents(path: Path):
    """Carga documentos desde un JSON; si no existe usa los de demostracion."""
    if not path.exists():
        print(f"\u2139\ufe0f No se encontro {path}; se usaran documentos de demostracion.")
        raw = DEMO_DOCUMENTS
    else:
        raw = json.loads(path.read_text(encoding="utf-8"))
    docs = []
    for d in raw:
        for campo in ("id", "title", "content"):
            if campo not in d:
                raise ValueError(f"Documento sin campo obligatorio '{campo}': {d}")
        d.setdefault("metadata", {})
        docs.append(d)
    print(f"\u2705 {len(docs)} documentos disponibles")
    return docs

# A2: ruta corregida al corpus propio.
RUTA_CORPUS = Path("data/company_documents.json")
if not RUTA_CORPUS.exists():
    RUTA_CORPUS = Path("company_documents.json")  # fallback si subiste solo el archivo

company_documents = load_documents(RUTA_CORPUS)

## Seccion 4 · Construir el pipeline RAG  _(Tareas A3 y A4)_

`build_pipeline` hace cinco cosas: fragmentar, embeddings, indexar en Qdrant,
crear el retriever y armar la cadena LCEL.

**Cambio A4:** la funcion ahora devuelve tambien el `vectorstore`, para poder
pedir scores de similitud desde `chat()`.

**Cambio A3 (prompt):** mas abajo defino `PROMPT_ANTES` y `PROMPT_DESPUES`. El
prompt *despues* es mas estricto: exige citar la fuente, responder solo con el
contexto, y decir explicitamente que no tiene la informacion cuando el contexto
no alcanza (clave para la tarea A1).

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

# ---- A3 · Ingenieria del prompt --------------------------------------------
PROMPT_ANTES = """Responde la pregunta del usuario usando el contexto.

Contexto:
{context}

Pregunta: {question}
Respuesta:"""

PROMPT_DESPUES = """Eres el asistente interno de la empresa. Responde la pregunta
usando UNICAMENTE la informacion del contexto. Reglas:
- Si el contexto no contiene la respuesta, di exactamente: "No tengo esa
  informacion en los documentos disponibles." No inventes datos.
- Cita entre parentesis el titulo del documento del que sacas cada dato.
- Responde en el idioma de la pregunta, de forma breve y concreta.

Contexto:
{context}

Pregunta: {question}
Respuesta:"""

# Cambia esta variable para comparar antes/despues en la tarea A3.
SYSTEM_PROMPT = PROMPT_DESPUES


def format_docs(docs):
    return "\n\n".join(
        f"[{d.metadata.get('title', 'sin titulo')}] {d.page_content}" for d in docs
    )


def build_pipeline(documents):
    # 4.1 Fragmentar en chunks
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=400, chunk_overlap=40,
        separators=["\n\n", "\n", ".", " "],
    )
    chunks = []
    for doc in documents:
        for i, frag in enumerate(splitter.split_text(doc["content"])):
            meta = {"source_id": doc["id"], "title": doc["title"], "chunk_index": i}
            meta.update(doc.get("metadata", {}))
            chunks.append(Document(page_content=frag, metadata=meta))

    # 4.2 Embeddings con NVIDIA NIM (la dimension se descubre, no se escribe a mano)
    embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, truncate="END")
    embedding_dim = len(embeddings.embed_query("prueba de dimension"))

    # 4.3 Indice vectorial en Qdrant (en memoria, distancia coseno)
    qdrant_client = QdrantClient(":memory:")
    qdrant_client.create_collection(
        collection_name=COLLECTION,
        vectors_config=VectorParams(size=embedding_dim, distance=Distance.COSINE),
    )
    vectorstore = QdrantVectorStore(
        client=qdrant_client, collection_name=COLLECTION, embedding=embeddings,
    )
    vectorstore.add_documents(chunks)

    # 4.4 Retriever: hasta 3 chunks, descarta score coseno < 0.3
    retriever = vectorstore.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": 3, "score_threshold": 0.3},
    )

    # 4.5 Generacion con LCEL
    llm = ChatNVIDIA(model=GENERATION_MODEL, temperature=0.2, max_completion_tokens=700)
    prompt = ChatPromptTemplate.from_template(SYSTEM_PROMPT)

    chain = (
        RunnableParallel(question=RunnablePassthrough(), documents=retriever)
        .assign(
            answer=(lambda x: {"context": format_docs(x["documents"]), "question": x["question"]})
            | prompt | llm | StrOutputParser()
        )
    )

    print("Construyendo el indice...")
    print(f"\u2705 Pipeline listo: {len(documents)} documentos, {len(chunks)} chunks")
    # A4: devolvemos tambien el vectorstore para calcular scores en chat().
    return chain, vectorstore, len(chunks)


chain, vectorstore, n_chunks = build_pipeline(company_documents)

## Secciones 5 y 6 · Interfaz Gradio y ejecucion  _(Tarea A4)_

`chat()` invoca el pipeline y arma la lista de fuentes.

**Cambio A4:** ademas del titulo, cada fuente muestra su **score de similitud
coseno**, obtenido con `vectorstore.similarity_search_with_relevance_scores`.
Asi se ve que tan relevante fue cada chunk recuperado.

In [ ]:
import gradio as gr


def fuentes_con_score(pregunta, k=3):
    """A4: devuelve [(titulo, score)] unico por titulo, ordenado por score."""
    pares = vectorstore.similarity_search_with_relevance_scores(pregunta, k=k)
    mejor_por_titulo = {}
    for doc, score in pares:
        titulo = doc.metadata.get("title", "sin titulo")
        if titulo not in mejor_por_titulo or score > mejor_por_titulo[titulo]:
            mejor_por_titulo[titulo] = score
    return sorted(mejor_por_titulo.items(), key=lambda t: t[1], reverse=True)


def chat(pregunta, history=None):
    try:
        resultado = chain.invoke(pregunta)
        respuesta = resultado["answer"]
        lineas = [f"- {t}  ·  score {s:.3f}" for t, s in fuentes_con_score(pregunta)]
        fuentes = "\n".join(lineas) if lineas else "_(sin fuentes por encima del umbral)_"
        return respuesta, f"### Fuentes (score de similitud)\n{fuentes}"
    except Exception as e:  # muestra el error sin romper la app
        return "Ocurrio un error al procesar la pregunta.", f"**Error:** {e}"


with gr.Blocks(title="RAG empresarial - NebulaCRM") as demo:
    gr.Markdown("# Asistente RAG empresarial\nPregunta sobre politicas y productos internos.")
    entrada = gr.Textbox(label="Tu pregunta", placeholder="Ej: cuanto cuesta el plan Enterprise?")
    boton = gr.Button("Preguntar", variant="primary")
    salida = gr.Markdown(label="Respuesta")
    panel_fuentes = gr.Markdown(label="Fuentes")
    boton.click(chat, inputs=entrada, outputs=[salida, panel_fuentes])
    entrada.submit(chat, inputs=entrada, outputs=[salida, panel_fuentes])

In [ ]:
# A1: lanzar la app. share=True crea una URL publica temporal.
demo.queue().launch(share=True, debug=True, show_error=True)

In [ ]:
# Ejecuta esta celda para cerrar instancias previas antes de relanzar.
import gradio as gr
gr.close_all()

## A1 · Prueba de punta a punta (incluida la pregunta trampa)

Estas comprobaciones no dependen de la UI; invocan el pipeline directamente.
Las cuatro primeras deben acertar con el corpus propio; la quinta esta **fuera
del corpus** y el prompt A3 debe hacer que el modelo admita que no sabe.

In [ ]:
preguntas_prueba = [
    "Cuanto cuesta el plan Enterprise de NebulaCRM?",
    "Cuantos dias por semana se puede trabajar remotamente?",
    "Que pasa con los reembolsos mayores a $100?",
    "Que ocurre durante la primera semana de onboarding?",
    "Cual es la politica de mascotas en la oficina?",  # <- fuera del corpus (A1)
]

for p in preguntas_prueba:
    r = chain.invoke(p)
    print("P:", p)
    print("R:", r["answer"].strip())
    print("Fuentes:", [t for t, _ in fuentes_con_score(p)])
    print("-" * 70)

## A3 · Comparacion antes / despues del prompt

Para la evidencia de A3: ejecuta el bloque siguiente, que reconstruye la cadena
con `PROMPT_ANTES` y con `PROMPT_DESPUES` y compara la respuesta a la **pregunta
trampa**. El prompt *antes* tiende a inventar o divagar; el *despues* responde
"No tengo esa informacion...". Pega ambas salidas en `RESPUESTAS.md`.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

def responder_con_prompt(texto_prompt, pregunta):
    llm = ChatNVIDIA(model=GENERATION_MODEL, temperature=0.2, max_completion_tokens=700)
    prompt = ChatPromptTemplate.from_template(texto_prompt)
    docs = vectorstore.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": 3, "score_threshold": 0.3},
    ).invoke(pregunta)
    cadena = (lambda _: {"context": format_docs(docs), "question": pregunta}) | prompt | llm | StrOutputParser()
    return cadena.invoke(None)

trampa = "Cual es la politica de mascotas en la oficina?"
print("=== PROMPT ANTES ===")
print(responder_con_prompt(PROMPT_ANTES, trampa).strip())
print("\n=== PROMPT DESPUES ===")
print(responder_con_prompt(PROMPT_DESPUES, trampa).strip())